# Milestone 2 — Hierarchical Parsing and Standardization

**CSC14119 Introduction to Data Science — Group 3**

This notebook is the entry point of Milestone 2. *Run all* executes the whole pipeline of the
`src/m2` package on the Milestone 1 corpus and shows the results:

1. configuration and environment (Google Colab **CPU** runtime, the benchmark testbed, or a local machine);
2. Colab only: mount Drive, pack `data/` into one tar on Drive and extract it to local disk (timed separately);
3. unit tests and the gold-set evaluation (accuracy of roots, headings, sentences, `\bibitem` conversion, reference dedup);
4. a 20-paper smoke test, then the **full run** (resumable) — one hierarchical tree per version, cleaned and
   normalized content, unified BibTeX entries, deduplication across versions;
5. compaction, structural validation and statistics;
6. results: parsing success, version coverage, counts before/after deduplication, element kinds,
   bibliography sources, accuracy proxies, running time and memory footprint, and a look at the tables.

Outputs (Parquet): `elements`, `hierarchy`, `bibtex` in `OUT`, internal working data and report exports in `WORK`.
The parser reads only `.tex`/`.bib` files and `metadata.json`; it never opens `references.json` and makes no
network call.

## 1. Configuration

In [ ]:
# Edit these values if needed ---------------------------------------------------------------
REPO_URL     = "https://github.com/DuyNhat1705/Intro2DS-Group3-Source.git"
BRANCH       = "ms2-parser"
DRIVE_DATA   = "/content/drive/MyDrive/Intro2DS-Group3/data"        # M1 corpus on Google Drive
DRIVE_TAR    = "/content/drive/MyDrive/Intro2DS-Group3/m2_data.tar"  # created once, then reused
DRIVE_OUT    = "/content/drive/MyDrive/Intro2DS-Group3/m2_output"    # results copied back here

WORKERS      = "auto"    # 'auto' = min(CPUs, free RAM / per-worker estimate); or an integer
RUN_FULL     = True      # False: only the smoke test
LIMIT        = None      # e.g. 500 to benchmark a subset; None = every paper
RUN_TESTS    = True      # unit tests + gold-set evaluation (about one minute)
RUN_ANYSTYLE = False     # optional bibitem refinement (needs Ruby + anystyle-cli), off by default
N_SHARDS     = 8         # Parquet shards per table after compaction
COPY_BACK    = True      # Colab: copy outputs to DRIVE_OUT

In [ ]:
import os, sys, time, json, shutil, subprocess, platform
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO = Path("/content/Intro2DS-Group3-Source")
    DATA = Path("/content/data")
    OUT, WORK = Path("/content/m2_out/tables"), Path("/content/m2_out/m2")
    SMOKE = Path("/content/m2_smoke")
else:  # local run: the notebook lives in src/, the corpus in data/
    REPO = Path.cwd() if (Path.cwd() / "src" / "m2").is_dir() else Path.cwd().parent
    DATA = Path(os.environ.get("M2_NB_DATA", REPO / "data"))
    OUT = Path(os.environ.get("M2_NB_OUT", REPO / "data" / "tables"))
    WORK = Path(os.environ.get("M2_NB_WORK", REPO / "data" / "m2"))
    SMOKE = Path(os.environ.get("M2_NB_SMOKE", REPO / "data" / "m2" / "smoke"))
    if os.environ.get("M2_NB_LIMIT"):              # used to test the notebook on a small sample
        LIMIT = int(os.environ["M2_NB_LIMIT"])

def sh(*args, check=True):
    """Run a command from the repository root with this kernel's Python and stream its output."""
    cmd = [str(a) for a in args]
    print("$", " ".join(cmd), flush=True)
    env = dict(os.environ, PYTHONIOENCODING="utf-8")
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         encoding="utf-8", errors="replace", env=env)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    if check and p.returncode:
        raise RuntimeError(f"command failed with exit code {p.returncode}")
    return p.returncode

timing = {}
print("Colab:", IN_COLAB, "| Python", platform.python_version(), "| repository:", REPO)
print("data:", DATA, "| tables:", OUT, "| working data:", WORK)

## 2. Environment

In [ ]:
if IN_COLAB and not REPO.exists():
    subprocess.run(["git", "clone", "-b", BRANCH, "--depth", "1", REPO_URL, str(REPO)], check=True)
if IN_COLAB:
    sh(sys.executable, "-m", "pip", "install", "-q", "-r", "src/requirements.txt")
import psutil
vm = psutil.virtual_memory()
print(f"CPUs: {os.cpu_count()} | RAM total {vm.total / 2**30:.1f} GB, available {vm.available / 2**30:.1f} GB"
      f" | disk free {shutil.disk_usage(str(REPO)).free / 2**30:.1f} GB")

## 3. Data (Colab only)

Parsing tens of thousands of small files directly from the Drive mount is very slow, so `data/` is packed once
into a tar on Drive (without `references.json`, which the parser never reads) and extracted to the local disk.
This step is timed separately from parsing.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    t0 = time.perf_counter()
    if not os.path.exists(DRIVE_TAR):
        src = Path(DRIVE_DATA)
        subprocess.run(["tar", "-cf", DRIVE_TAR, "--exclude=*/references.json", "--exclude=tables",
                        "--exclude=m2", "--exclude=run_logs", "--exclude=report_exports",
                        "-C", str(src.parent), src.name], check=True)
    timing["tar_create_s"] = time.perf_counter() - t0
    t0 = time.perf_counter()
    if not DATA.exists():
        subprocess.run(["tar", "-xf", DRIVE_TAR, "-C", "/content"], check=True)
    timing["tar_extract_s"] = time.perf_counter() - t0
n_papers = sum(1 for p in DATA.iterdir() if p.is_dir() and not p.name.startswith(".")
               and p.name not in ("tables", "m2", "run_logs", "report_exports"))
print("paper folders:", n_papers, "|", {k: round(v, 1) for k, v in timing.items()})

## 4. Tests and gold-set evaluation

Unit tests use small inline LaTeX/BibTeX snippets. The gold sets (Python modules in `src/m2/tests/`) were
checked by hand on real papers: 29 versions (root, compilation path, 1,197 headings), 379 sentence
boundaries, 152 `\bibitem`s and 97 reference pairs. Targets: root and compilation path 100 %, headings ≥ 95 %,
sentence F1 ≥ 0.97, bibitem title ≥ 90 % and year ≥ 95 %, reference dedup precision ≥ 98 % and recall ≥ 95 %.

In [ ]:
if RUN_TESTS:
    sh(sys.executable, "-m", "pytest", "src/m2/tests", "-q", "-p", "no:cacheprovider")
    sh(sys.executable, "-m", "src.m2.tests.eval_gold", "--data-dir", DATA,
       "--json", WORK / "report_exports" / "gold_eval.json")

## 5. Smoke test (20 papers, 2 workers)

In [ ]:
shutil.rmtree(SMOKE, ignore_errors=True)
sh(sys.executable, "-m", "src.m2.run", "--data-dir", DATA, "--out", SMOKE / "tables",
   "--work-dir", SMOKE / "m2", "--workers", "2", "--limit", 20, "--no-progress")
sh(sys.executable, "-m", "src.m2.validate", "--out", SMOKE / "tables", "--work-dir", SMOKE / "m2",
   "--data-dir", DATA)
shutil.rmtree(SMOKE, ignore_errors=True)

## 6. Full run

One process per worker with a watchdog (60 s per paper, killed and retried once in safe mode), largest papers
first, per-worker Parquet shards. The run is **resumable**: running this cell again skips every paper whose
checkpoint signature (relative paths + sizes + contents + parser version) is unchanged.

In [ ]:
args = [sys.executable, "-m", "src.m2.run", "--data-dir", DATA, "--out", OUT, "--work-dir", WORK,
        "--workers", WORKERS, "--resume", "--no-progress"]
if not RUN_FULL:
    args += ["--limit", 20]
elif LIMIT:
    args += ["--limit", LIMIT]
if RUN_ANYSTYLE:
    if IN_COLAB:
        subprocess.run("apt-get -qq install -y ruby-full > /dev/null && gem install -q anystyle-cli",
                       shell=True, check=False)
    args += ["--refine-bibitem"]
t0 = time.perf_counter()
sh(*args)
timing["parse_wall_s"] = time.perf_counter() - t0

## 7. Compaction, validation and statistics

In [ ]:
t0 = time.perf_counter()
sh(sys.executable, "-m", "src.m2.run", "--out", OUT, "--work-dir", WORK, "--compact-only", "--n-shards", N_SHARDS)
timing["compact_s"] = time.perf_counter() - t0
rc = sh(sys.executable, "-m", "src.m2.validate", "--out", OUT, "--work-dir", WORK, "--data-dir", DATA, check=False)
print("validation exit code:", rc, "(0 = no structural violation)")
sh(sys.executable, "-m", "src.m2.stats", "--out", OUT, "--work-dir", WORK)

## 8. Results

In [ ]:
import pandas as pd
import pyarrow as pa, pyarrow.parquet as pq
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 120)
EXP = WORK / "report_exports"
summary = json.load(open(EXP / "summary.json", encoding="utf-8"))
val = json.load(open(EXP / "validation.json", encoding="utf-8"))
runs = [json.load(open(p, encoding="utf-8")) for p in sorted((WORK / "runs").glob("*.json"))]
run = max(runs, key=lambda r: r.get("done", 0))  # the run that parsed the most papers

def read_table(d, columns=None):
    files = sorted(Path(d).glob("*.parquet"))
    return pa.concat_tables([pq.read_table(f, columns=columns) for f in files]).unify_dictionaries().to_pandas()

### 8.1 Parsing success and version coverage

In [ ]:
status = pd.read_csv(EXP / "status_counts.csv").set_index("status")
display(status.T)
print(f"success rate on versions with TeX (ok / root_fallback / partial): {summary['success_rate_non_empty']:.4%}"
      f" | failed: {summary['failed_rate_non_empty']:.4%}")
cov = pd.read_csv(EXP / "version_coverage.csv")
print("publications:", len(cov), "| versions per publication:", cov["n_versions"].value_counts().sort_index().to_dict())
print("publications with at least one parsed version:",
      int(((cov["ok"] + cov["root_fallback"] + cov["partial"]) > 0).sum()),
      "| all versions empty:", int((cov["empty"] == cov["n_versions"]).sum()))
display(cov.head(10))
reasons = pd.read_csv(EXP / "failure_reasons.csv")
display(reasons.sort_values("versions", ascending=False).head(10))

### 8.2 Counts before and after deduplication

In [ ]:
d = summary["dedup"]
dedup = pd.DataFrame({
    "before": [d["nodes_before"], d["bib_cited_before_dedup"]],
    "after": [d["elements_after"], d["bib_after"]]}, index=["hierarchy elements", "BibTeX entries"])
dedup["reduction"] = 1 - dedup["after"] / dedup["before"]
display(dedup)
print("raw BibTeX entries in .bib / \\bibitem (before keeping only cited ones):", d["bib_raw"])
print("leaves repeating their content inside one version (kept as separate elements):",
      d["leaf_repeats_within_version"], "| identical versions reused:", d["reused_versions"])

### 8.3 Elements, depth and bibliography sources

In [ ]:
kinds = pd.read_csv(EXP / "kind_counts.csv").sort_values("elements", ascending=False)
depth = pd.read_csv(EXP / "depth_hist.csv")
src = pd.read_csv(EXP / "bib_sources.csv")
meth = pd.read_csv(EXP / "bib_methods.csv")
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ax[0].barh(kinds["kind"], kinds["elements"]); ax[0].set_xscale("log"); ax[0].set_title("elements by kind")
dd = depth.groupby("depth")["count"].sum()
ax[1].bar(dd.index, dd.values); ax[1].set_yscale("log"); ax[1].set_title("nodes by depth")
ax[2].bar(meth["parse_method"], meth["entries"]); ax[2].set_title("BibTeX entries by parse method")
plt.tight_layout(); plt.show()
files = pd.read_csv(EXP / "files.csv")
print("compiled .tex files:", int(files["n_compile_files"].sum()), "| redundant .tex files:",
      int(files["n_redundant_files"].sum()), "| missing includes:", int(files["n_missing_includes"].sum()))
display(files["bib_source"].value_counts().rename("versions").to_frame())

### 8.4 Quality checks and accuracy proxies

In [ ]:
checks = {k: val.get(k) for k in ("violations_total", "residual_latex_rate", "cite_coverage",
                                   "cite_coverage_papers_with_bib_source", "abstract_found_share",
                                   "abstract_fidelity_median", "abstract_fidelity_mean", "title_fidelity_median",
                                   "section_cmds", "section_nodes")}
display(pd.Series(checks, name="value").to_frame())
display(pd.Series(val["violations"], name="violations").to_frame().T)
fid = pd.read_csv(EXP / "fidelity.csv")
print("lowest abstract fidelity (debugging queue):")
display(fid.sort_values("abstract_fidelity").head(8))
gold = EXP / "gold_eval.json"
if gold.exists():
    g = json.load(open(gold, encoding="utf-8"))
    rows = {"root": g["structure"]["root_acc"], "compilation path": g["structure"]["compile_acc"],
            "headings": g["structure"]["heading_acc"], "sentence F1": g["sentences"]["rules"]["f1"],
            "bibitem title": g["bibitems"]["plain"]["title_acc"], "bibitem year": g["bibitems"]["plain"]["year_acc"],
            "ref-dedup precision": g["pairs"]["default"]["precision"],
            "ref-dedup recall": g["pairs"]["default"]["recall"]}
    display(pd.Series(rows, name="gold-set score").to_frame())

### 8.5 Running time and memory footprint

In [ ]:
perf = {k: run.get(k) for k in ("papers", "workers", "done", "skipped", "failed", "timeouts", "wall_s",
                                 "paper_ms_mean", "paper_ms_median", "paper_ms_p95", "version_ms_mean",
                                 "version_ms_median", "version_ms_p95", "peak_rss_mb", "peak_workers_rss_mb",
                                 "paragraph_cache_hit_rate", "reused_versions")}
perf.update({k: round(v, 1) for k, v in timing.items()})
perf["output_MB (elements+hierarchy+bibtex)"] = round(sum(
    f.stat().st_size for t in ("elements", "hierarchy", "bibtex") for f in (OUT / t).glob("*.parquet")) / 2**20, 2)
perf["working_data_MB"] = round(sum(f.stat().st_size for f in WORK.rglob("*") if f.is_file()) / 2**20, 2)
display(pd.Series(perf, name="value").to_frame())
tm = pd.read_csv(EXP / "timing.csv")
stage = tm[["gather_ms", "preprocess_ms", "hierarchy_ms", "bib_ms", "dedup_ms"]].sum() / 1000
display(stage.rename("total seconds per stage").to_frame().T)
json.dump(dict(perf, colab=IN_COLAB, cpus=os.cpu_count()), open(EXP / "benchmark.json", "w"), indent=1)

### 8.6 A look at the tables

In [ ]:
el = read_table(OUT / "elements")
hi = read_table(OUT / "hierarchy")
bi = read_table(OUT / "bibtex")
print("rows: elements", len(el), "| hierarchy", len(hi), "| bibtex", len(bi))
pid = hi.groupby("arxiv_id")["version"].nunique().sort_values(ascending=False).index[0]  # a multi-version paper
print("example publication:", pid)
display(el[el.arxiv_id == pid].head(12))
display(hi[hi.arxiv_id == pid].head(12))
display(bi[bi.arxiv_id == pid].head(5))
sh(sys.executable, "-m", "src.m2.inspect_tree", pid, "--version", 1, "--depth", 2, "--width", 100,
   "--data-dir", DATA)

## 9. Copy the outputs back to Drive (Colab)

In [ ]:
if IN_COLAB and COPY_BACK:
    dst = Path(DRIVE_OUT)
    dst.mkdir(parents=True, exist_ok=True)
    for s, name in ((OUT, "tables"), (WORK, "m2")):
        shutil.copytree(s, dst / name, dirs_exist_ok=True)
    print("copied to", dst)